# Amatrice paper experiment guide (LSTM / coherence-source comparison)

This notebook is a dedicated experiment guide for the workflow you described:

1. Crop the Amatrice interferogram pairs to `-l 42.6 42.7 -L 13.2 13.4`.
2. Generate auxiliary INT-derived products:
   - `unfilt_fine.cor`
   - `underamp_unfilt_fine.cor`
   - `underamp_unfilt_fine_circ.cor`
   - `filt_fine.std`
3. Build separate datasets for:
   - `fine.cor.full` (band 2 coherence)
   - `filt_fine.cor`
   - `unfilt_fine.cor`
   - `underamp_unfilt_fine.cor`
   - `underamp_unfilt_fine_circ.cor`
   - `filt_fine.std`
4. Train LSTM models with and without timestamp features.
5. Compute **non-zscore** NDI scores.

> Important:
> - Coherence score uses `(pred - obs) / (pred + obs + eps)`.
> - Phase-STD score uses `(obs - pred) / (pred + obs + eps)`.
> - Therefore `filt_fine.std` must be trained/scored with `--timeseries-metric phase_std`.


In [ ]:
from pathlib import Path
import subprocess

BASE_DIR = Path('/data6/WORKDIR/AmatriceSenDT22/merged/interferograms')
GEOM_REF_DIR = Path('/data6/WORKDIR/AmatriceSenDT22/merged/geom_reference')
CROPPED_DIR = BASE_DIR / 'cropped_paper_bbox'
EVENT_DATE = '20160824'
NEXT_DATE = '20160821_20160914'
LAT_MIN, LAT_MAX = 42.6, 42.7
LON_MIN, LON_MAX = 13.2, 13.4

PAIR_DIRS = [
    '20160306_20160330',
    '20160330_20160517',
    '20160517_20160529',
    '20160529_20160610',
    '20160610_20160704',
    '20160704_20160716',
    '20160716_20160728',
    '20160728_20160809',
    '20160809_20160821',
    '20160821_20160914',
]


def run_cmd(cmd: str, check: bool = True):
    print('>>>', cmd)
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        if check:
            raise RuntimeError(f'Command failed: {cmd}')
    return proc


## 1. Crop the requested pairs to the paper bbox


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step crop "
    f"--base-dir {BASE_DIR} "
    f"--geom-reference-dir {GEOM_REF_DIR} "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-min {LAT_MIN} --lat-max {LAT_MAX} "
    f"--lon-min {LON_MIN} --lon-max {LON_MAX}"
)


## 2. Generate INT-derived auxiliary products


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step prepare_int_aux "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--aux-corr-win 5 --aux-phsig-win 5 "
    f"--aux-variance-win 5 --aux-variance-looks 3.0 "
    f"--aux-block-lines 512"
)


## 3. Inspect prepared products


In [ ]:
run_cmd(f"find {CROPPED_DIR} -maxdepth 1 -type f | sort")


## 4. Build all requested datasets


In [ ]:
DATASET_SPECS = [
    ('fine.cor.full', 'coherence', 'dataset_rnn_fine_cor_full'),
    ('filt_fine.cor', 'coherence', 'dataset_rnn_filt_fine_cor'),
    ('unfilt_fine.cor', 'coherence', 'dataset_rnn_unfilt_fine_cor'),
    ('underamp_unfilt_fine.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_cor'),
    ('underamp_unfilt_fine_circ.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_circ_cor'),
    ('filt_fine.std', 'phase_std', 'dataset_rnn_filt_fine_std'),
]

for observation_file, metric, dataset_name in DATASET_SPECS:
    extra = "--looks 3.0" if observation_file == 'filt_fine.std' else ""
    run_cmd(
        f"python -m insar_pipeline.app --step build_dataset "
        f"--cropped-dir {CROPPED_DIR} "
        f"--output-dir {CROPPED_DIR} "
        f"--event-date {EVENT_DATE} "
        f"--input-source cor "
        f"--observation-file {observation_file} "
        f"--dataset-name {dataset_name} {extra}"
    )


## 5. LSTM training: compare timestamp on/off


In [ ]:
EXPERIMENTS = [
    ('fine.cor.full', 'coherence', 'dataset_rnn_fine_cor_full'),
    ('filt_fine.cor', 'coherence', 'dataset_rnn_filt_fine_cor'),
    ('unfilt_fine.cor', 'coherence', 'dataset_rnn_unfilt_fine_cor'),
    ('underamp_unfilt_fine.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_cor'),
    ('underamp_unfilt_fine_circ.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_circ_cor'),
    ('filt_fine.std', 'phase_std', 'dataset_rnn_filt_fine_std'),
]

for observation_file, metric, dataset_name in EXPERIMENTS:
    for timestamp_flag in ['time', 'notime']:
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        artifact_tag = observation_file.replace('.', '_')
        run_cmd(
            f"python -m insar_pipeline.app --step train_predict "
            f"--dataset-dir {CROPPED_DIR / dataset_name} "
            f"--output-dir {CROPPED_DIR} "
            f"--next-date {NEXT_DATE} "
            f"--timeseries-metric {metric} "
            f"--ts-model lstm "
            f"--artifact-tag {artifact_tag} {disable}"
        )


## 6. Compute non-zscore NDI scores


In [ ]:
for observation_file, metric, dataset_name in EXPERIMENTS:
    for timestamp_flag in ['time', 'notime']:
        artifact_tag = observation_file.replace('.', '_')
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        run_cmd(
            f"python -m insar_pipeline.app --step score "
            f"--dataset-dir {CROPPED_DIR / dataset_name} "
            f"--predict-dir {CROPPED_DIR / 'predict'} "
            f"--output-dir {CROPPED_DIR} "
            f"--timeseries-metric {metric} "
            f"--score-mode ndi "
            f"--ts-model lstm "
            f"--artifact-tag {artifact_tag} {disable}"
        )


## 7. Optional geocoded outputs for selected scores


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step output "
    f"--predict-dir {CROPPED_DIR / 'predict'} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-file {CROPPED_DIR / 'lat_cropped.rdr'} "
    f"--lon-file {CROPPED_DIR / 'lon_cropped.rdr'} "
    f"--subset-params '-l 42.6 42.7 -L 13.2 13.4'"
)


## 8. Suggested result table export


In [ ]:
run_cmd(f"find {CROPPED_DIR / 'predict'} -maxdepth 1 -type f | sort")
